<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "598"
iso_country = "Papua New Guinea"
year = 2024
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 30.800*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Papua New Guinea': 

Number of trade records present in dataset: 31539
Number of partner countries present in dataset: 156
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$5554.7m
Total weight within the dataset: 4878.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,598,Papua New Guinea,156,China,Import,Sea,846211,846211 | Machine-tools; forgi,8462,8462 | Machine-tools (inclu,84,84 | Machinery and mechan,NaN,NaN,NaN,NaN,NaN,9.321420,NaN,9.321420,NaN,0.00000,NaN,1.258392,6299.0,3914.015929,4.925365,['all'],['all'],0.022957,0.022934,0.022935,0.022935,0.022946,0.023026,0.023188,0.037242,0.024341,0.023315,0.023346,0.023962,0.022765,0.035931,0.025288,0.023972,0.023343,0.024368,0.022475,0.039390,0.029746,0.027318,0.025500,0.025304,0.022191,0.043703,0.034786,0.032697,0.026762,0.026908,0.021902,0.043855,0.041309,0.042625,0.028504,0.028125,0.021569,0.041888,0.040223,0.043045,0.033965,0.028572,0.021265,0.038425,0.039765,0.041934,0.033571,0.029343,0.020875,0.035934,0.036398,0.04174,0.034707,0.029428,nan
1,1,598,Papua New Guinea,156,China,Import,Sea,846219,846219 | Machine-tools; hot f,8462,8462 | Machine-tools (inclu,84,84 | Machinery and mechan,NaN,NaN,NaN,NaN,NaN,82097.929054,NaN,82097.929054,u,1259.44278,8629.553044,11083.220422,6299.0,3914.015929,43379.901277,['all'],['all'],202.195622,201.988012,202.002963,202.002963,202.099423,202.801032,204.223176,328.003476,214.382272,205.349696,205.620258,211.040766,200.503334,316.463070,222.724735,211.129827,205.595979,214.619564,197.951247,346.928732,261.987246,240.604734,224.585868,222.867311,195.444135,384.915286,306.379664,287.972914,235.703682,236.991939,192.898224,386.248236,363.830608,375.419896,251.046035,247.713893,189.970592,368.926560,354.263498,379.112388,299.149336,251.644219,187.288199,338.428405,350.230337,369.335797,295.671187,258.435714,183.852443,316.488108,320.569766,367.62072,305.684440,259.184198,nan


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $918.4m (16.5%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               324816946.5  812824736.7      5.8     16.7
21 | Miscellaneous edible  144922206.2   34405664.0      2.6      0.7
02 | Meat and edible meat   84020108.3   28352420.0      1.5      0.6
19 | Preparations of cere   48838923.4   22098436.9      0.9      0.5
17 | Sugars and sugar con   48583179.4   94878284.1      0.9      1.9
23 | Food industries, res   48466861.5   50553360.2      0.9      1.0
16 | Meat, fish, crustace   34861705.3    7421342.8      0.6      0.2
15 | Animal, vegetable or   34787283.4   30005163.9      0.6      0.6
04 | Dairy produce; birds   27906015.5   17609499.0      0.5      0.4
20 | Preparations of vege   25199153.8   18652830.2      0.5      0.4
11 | Products of the mill   21528051.4   45922026.3      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $766.5m (13.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       392827137.2  474509137.8      7.1      9.7
Food & Beverages  366866757.6  237958696.8      6.6      4.9
Fishing             6847711.8    2927641.6      0.1      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $749.0m (13.5%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
OFD - Food products n.e.c.                    330392843.5  155963491.5   
PCR - Processed rice                          192175393.9  333808659.2   
CMT - Bovine meat prods                        49513177.9   10412641.5   
OMT - Meat products n.e.c.                     46209508.2   23971954.5   
VOL - Vegetable oils and fats                  42972482.4   52277320.3   
SGR - Sugar                                    40632322.0   89100324.1   
MIL - Dairy products                           23664779.3   11171170.4   
V_F - Vegetables, fruit, nuts                  11697893.9   13613894.6   
OAP - Animal products n.e.c.                    6295376.0    1502020.8   
OCR - Crops n.e.c.                              3941437.2    1293193.1   
PDR - Paddy rice                              

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 16.5% 
GTAP 13.5% 
EORA 13.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27844/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,598,Papua New Guinea,156,China,Import,Sea,846211,846211 | Machine-tools; forgi,8462,8462 | Machine-tools (inclu,84,84 | Machinery and mechan,NaN,NaN,NaN,NaN,NaN,9.321420,NaN,9.321420,NaN,0.00000,NaN,1.258392,6299.0,3914.015929,4.925365,['all'],['all'],0.022957,0.022934,0.022935,0.022935,0.022946,0.023026,0.023188,0.037242,0.024341,0.023315,0.023346,0.023962,0.022765,0.035931,0.025288,0.023972,0.023343,0.024368,0.022475,0.039390,0.029746,0.027318,0.025500,0.025304,0.022191,0.043703,0.034786,0.032697,0.026762,0.026908,0.021902,0.043855,0.041309,0.042625,0.028504,0.028125,0.021569,0.041888,0.040223,0.043045,0.033965,0.028572,0.021265,0.038425,0.039765,0.041934,0.033571,0.029343,0.020875,0.035934,0.036398,0.04174,0.034707,0.029428,nan,NaN
1,1,598,Papua New Guinea,156,China,Import,Sea,846219,846219 | Machine-tools; hot f,8462,8462 | Machine-tools (inclu,84,84 | Machinery and mechan,NaN,NaN,NaN,NaN,NaN,82097.929054,NaN,82097.929054,u,1259.44278,8629.553044,11083.220422,6299.0,3914.015929,43379.901277,['all'],['all'],202.195622,201.988012,202.002963,202.002963,202.099423,202.801032,204.223176,328.003476,214.382272,205.349696,205.620258,211.040766,200.503334,316.463070,222.724735,211.129827,205.595979,214.619564,197.951247,346.928732,261.987246,240.604734,224.585868,222.867311,195.444135,384.915286,306.379664,287.972914,235.703682,236.991939,192.898224,386.248236,363.830608,375.419896,251.046035,247.713893,189.970592,368.926560,354.263498,379.112388,299.149336,251.644219,187.288199,338.428405,350.230337,369.335797,295.671187,258.435714,183.852443,316.488108,320.569766,367.62072,305.684440,259.184198,nan,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $699.2m of $5554.7m (12.6%)
Volume: 1083.8 of 4878.8 million kilograms (22.2%)
Value-to-Weight: 0.65 from 1.14 US$ per kg (+-43.3%)
Average distance to market (volume-weighted): 3199.3 from 3326.3 nm (+-3.8%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    691  374041750.8  426947202.7      6.7      8.8
1  Unprocessed Cereals     26  131389136.6  476840630.8      2.4      9.8
2        Meat and Prep     47   69583890.1   27600745.6      1.3      0.6
3      Sugar and Honey    135   48611595.3   94886430.9      0.9      1.9
4                Dairy    114   27437630.2   17459112.4      0.5      0.4
5        Fats and Oils    163   22904567.0   16086072.9      0.4      0.3
6           Vegetables    404   17348543.9   16394661.5      0.3      0.3
7               Fruits    220    7896683.7    7582336.8      0.1      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,23950316.8,4.277521e+07,26212787.2,24538395.3,24609574.2,25195840.1,22126835.3,4.948576e+07,4.751013e+07,4.805227e+07,32080441.9,30563381.7,20851984.5,3.941721e+07,3.998648e+07,4.837324e+07,4.026260e+07,31722687.5
1,Fats and Oils,25295331.0,4.287560e+07,27274347.3,25617382.9,25684918.0,26413045.8,23597763.8,4.915940e+07,4.727790e+07,4.829335e+07,32905119.3,31407833.7,22398393.9,3.996737e+07,4.063424e+07,4.814289e+07,4.022629e+07,32519753.4
2,Fruits,10857146.8,1.932403e+07,11866702.3,11118396.2,11149753.2,11415413.0,10037954.6,2.235335e+07,2.144720e+07,2.170496e+07,14504767.4,13832532.5,9463604.6,1.784999e+07,1.810629e+07,2.185567e+07,1.816520e+07,14360716.2
3,Meat and Prep,35196910.1,6.296109e+07,38545558.1,36073284.6,36178526.1,37037506.8,32506296.7,7.278249e+07,6.991157e+07,7.069512e+07,47210346.3,44949187.7,30628250.3,5.798195e+07,5.882728e+07,7.122563e+07,5.924159e+07,46649639.7
4,Processed Cereals,745438469.6,1.309566e+09,811709032.8,761486647.8,763633168.1,782440833.7,690849726.1,1.521499e+09,1.458629e+09,1.479307e+09,993231764.6,942948685.7,652597077.3,1.215648e+09,1.232755e+09,1.481819e+09,1.235010e+09,977743944.7
5,Sugar and Honey,127226650.2,2.109363e+08,136307164.8,128640153.5,128986312.6,132582046.3,119110409.0,2.498318e+08,2.382024e+08,2.437880e+08,167312678.4,156080908.7,113436876.3,2.010322e+08,2.034599e+08,2.392141e+08,2.013256e+08,161003408.0
6,Unprocessed Cereals,635535400.7,1.136830e+09,695994656.6,651356425.7,653256645.9,668767765.1,586955226.3,1.314176e+09,1.262335e+09,1.276489e+09,852449765.3,811616085.7,553046219.2,1.046939e+09,1.062202e+09,1.286056e+09,1.069675e+09,842318353.6
7,Vegetables,24119353.3,4.108790e+07,25905520.5,24490673.3,24546099.6,25167571.8,22511565.0,4.791570e+07,4.562156e+07,4.655920e+07,31142031.8,30072132.7,21324216.6,3.861298e+07,3.917301e+07,4.626575e+07,3.845967e+07,31302952.8


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.69,0.08,0.02,0.02,0.05,1.00,0.93,0.94,0.36,0.31,0.68,0.70,1.00,0.71,0.40
1,Fats and Oils,0.77,0.09,0.01,0.02,0.05,1.12,1.03,1.08,0.41,0.34,0.77,0.80,1.12,0.78,0.44
2,Fruits,1.07,0.13,0.03,0.04,0.07,1.56,1.44,1.48,0.57,0.48,1.06,1.09,1.57,1.10,0.62
3,Meat and Prep,0.40,0.05,0.01,0.01,0.03,0.58,0.54,0.55,0.21,0.18,0.39,0.41,0.58,0.41,0.23
4,Processed Cereals,1.51,0.18,0.04,0.05,0.10,2.22,2.05,2.11,0.81,0.67,1.51,1.55,2.22,1.56,0.87
5,Sugar and Honey,1.72,0.19,0.03,0.04,0.11,2.69,2.45,2.56,0.99,0.76,1.80,1.85,2.59,1.81,0.98
6,Unprocessed Cereals,3.82,0.46,0.12,0.13,0.25,5.53,5.14,5.25,2.02,1.71,3.76,3.88,5.58,3.93,2.20
7,Vegetables,0.98,0.10,0.02,0.02,0.06,1.46,1.33,1.39,0.50,0.44,1.00,1.03,1.44,0.99,0.58
